In [3]:
# =====================================
#  Sinhala Preprocessing for PyCharm
# =====================================

import json
import pandas as pd
from tqdm import tqdm
import re
import os
from sklearn.model_selection import train_test_split

# Configuration - CHANGE THESE PATHS
INPUT_FILE = "sinhala_dataset.jsonl"  # Your input file path
OUTPUT_DIR = "processed_sinhala"      # Where to save processed files

MAX_PROMPT_TOKENS = 200
MAX_TARGET_TOKENS = 300
MIN_RESPONSE_LENGTH = 20

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_jsonl_safe(filepath):
    """Load JSONL file safely, skipping bad lines"""
    records = []
    bad_lines = 0

    print(f"Loading {filepath}...")

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                bad_lines += 1

    print(f"✓ Loaded {len(records)} rows, skipped {bad_lines} bad lines")
    return pd.DataFrame(records)

def clean_text(text):
    """Clean text by removing extra whitespace"""
    text = text.replace("\n", " ").replace("\t", " ")
    text = " ".join(text.split())
    return text.strip()

def estimate_tokens(text):
    """Estimate token count by splitting on whitespace"""
    return len(text.split())

def check_sinhala_script(text):
    """Check if text contains Sinhala characters"""
    return bool(re.search(r'[\u0D80-\u0DFF]', text))

def build_prompt(row):
    """Build prompt in the same format as English preprocessing"""
    emotion = row["user_emotion"]
    return (
        f"Context: The user is feeling {emotion}. "
        f"Task: Respond as an empathetic assistant.\n"
        f"User: {row['user_input']}\n"
        f"Assistant:"
    )

def main():
    """Main preprocessing pipeline"""

    print("="*70)
    print("SINHALA DATASET PREPROCESSING")
    print("="*70)

    # 1. Load data
    df = load_jsonl_safe(INPUT_FILE)
    df["language"] = "si"

    print(f"\nInitial rows: {len(df)}")

    # 2. Clean text
    print("\nCleaning text...")
    df["user_input"] = df["user_input"].apply(clean_text)
    df["bot_response"] = df["bot_response"].apply(clean_text)
    df["user_emotion"] = df["user_emotion"].str.lower().str.strip()

    # 3. Length filters
    print("Applying length filters...")
    df = df[df["user_input"].str.len() > 5]
    df = df[df["bot_response"].str.len() > MIN_RESPONSE_LENGTH]
    print(f"  After length filter: {len(df)} rows")

    # 4. Remove duplicates
    print("Removing duplicates...")
    df = df.drop_duplicates(subset=["user_input", "bot_response"])
    print(f"  After deduplication: {len(df)} rows")

    # 5. Script validation (check for Sinhala characters)
    print("Validating Sinhala script...")
    df["valid_script"] = df.apply(
        lambda row: check_sinhala_script(row["user_input"]) and
                    check_sinhala_script(row["bot_response"]),
        axis=1
    )
    df = df[df["valid_script"]].drop(columns=["valid_script"])
    print(f"  After script validation: {len(df)} rows")

    # 6. Create prompts (SAME FORMAT AS ENGLISH)
    print("\nCreating prompts...")
    df["prompt"] = df.apply(build_prompt, axis=1)
    df["target"] = df["bot_response"]

    # 7. Token filtering
    print("Filtering by token length...")
    df["prompt_tokens"] = df["prompt"].apply(estimate_tokens)
    df["target_tokens"] = df["target"].apply(estimate_tokens)
    df = df[
        (df["prompt_tokens"] < MAX_PROMPT_TOKENS) &
        (df["target_tokens"] < MAX_TARGET_TOKENS)
    ]
    print(f"  After token filter: {len(df)} rows")

    # 8. Show sample
    print("\n" + "="*70)
    print("SAMPLE PROMPT:")
    print("="*70)
    print(df["prompt"].iloc[0])
    print("\nTARGET:")
    print(df["target"].iloc[0])
    print("="*70)

    # 9. Split data (80/10/10)
    print("\nSplitting data...")
    train_df, temp_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df["user_emotion"]
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=42,
        stratify=temp_df["user_emotion"]
    )

    print(f"  Train: {len(train_df)} rows")
    print(f"  Val:   {len(val_df)} rows")
    print(f"  Test:  {len(test_df)} rows")

    # 10. Save files
    print("\nSaving files...")

    train_path = os.path.join(OUTPUT_DIR, "train_si.jsonl")
    val_path = os.path.join(OUTPUT_DIR, "val_si.jsonl")
    test_path = os.path.join(OUTPUT_DIR, "test_si.jsonl")

    train_df[["prompt", "target", "language", "user_emotion"]].to_json(
        train_path, orient="records", lines=True, force_ascii=False
    )
    val_df[["prompt", "target", "language", "user_emotion"]].to_json(
        val_path, orient="records", lines=True, force_ascii=False
    )
    test_df[["prompt", "target", "language", "user_emotion"]].to_json(
        test_path, orient="records", lines=True, force_ascii=False
    )

    print(f"  ✓ {train_path}")
    print(f"  ✓ {val_path}")
    print(f"  ✓ {test_path}")

    # 11. Summary
    print("\n" + "="*70)
    print("PREPROCESSING COMPLETE!")
    print("="*70)
    print(f"Total samples: {len(df)}")
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"\nOutput directory: {OUTPUT_DIR}")
    print("="*70)

    # 12. Show emotion distribution
    print("\nEmotion Distribution (Training Set):")
    print(train_df["user_emotion"].value_counts())

if __name__ == "__main__":
    main()

SINHALA DATASET PREPROCESSING
Loading sinhala_dataset.jsonl...
✓ Loaded 996 rows, skipped 4 bad lines

Initial rows: 996

Cleaning text...
Applying length filters...
  After length filter: 996 rows
Removing duplicates...
  After deduplication: 995 rows
Validating Sinhala script...
  After script validation: 995 rows

Creating prompts...
Filtering by token length...
  After token filter: 995 rows

SAMPLE PROMPT:
Context: The user is feeling angry. Task: Respond as an empathetic assistant.
User: ඒක මගේ පණිවිඩය ආයෙත් නොසලකා හැරියා කියලා මට විශ්වාස කරන්න බෑ!
Assistant:

TARGET:
ඒක කලකිරීමක් වගේ. ප්‍රතිචාර දක්වන්න කලින්, ගැඹුරු හුස්මක් ගන්න. එයාලා කාර්යබහුල වෙලා හෝ අවධානය වෙනතකට යොමු වෙලා ඉන්න පුළුවන්ද? ඒ අදහසට සහාය දෙන සාක්ෂි මොනවාද කියලා බලමු.

Splitting data...
  Train: 796 rows
  Val:   99 rows
  Test:  100 rows

Saving files...
  ✓ processed_sinhala\train_si.jsonl
  ✓ processed_sinhala\val_si.jsonl
  ✓ processed_sinhala\test_si.jsonl

PREPROCESSING COMPLETE!
Total samples: 995
Train: 7